In [2]:
# =============================================================================
# SUPERVISED FINE-TUNING (SFT) FOR CORRECTION-GPT
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHAT IS SFT? (picture first)
# ---------------------------------------------------------------------------
# Think of three stages of raising a model:
#
#   1) PRETRAIN (notebooks 7–8)
#      Read tons of text → learn language / next-token guessing.
#      Like teaching a kid to read and finish sentences.
#      Model: "The price is $" → might continue with anything plausible.
#
#   2) SUPERVISED FINE-TUNING  ← YOU ARE HERE
#      Show many (prompt → ideal answer) pairs and train it to IMITATE them.
#      Like flashcards: "When you see THIS situation, say THAT."
#      Model becomes a specialist for a task (here: quiet sales corrections).
#
#   3) Later (optional): RLHF / preference tuning
#      Rank answers good/bad. Not this notebook.
#
# Sticky cartoon for ONE training example:
#
#   PROMPT (what the model sees / must continue from)
#   ┌─────────────────────────────────────────────────────────┐
#   │ Instruction: You are an AI sales coach. Whisper a fix.  │
#   │ Ground truth: the price is $500 per month               │
#   │ User said:    the price is $300 per month               │
#   └─────────────────────────────────────────────────────────┘
#                              │
#                              ▼  model should generate →
#   TARGET OUTPUT
#   ┌─────────────────────────────────────────────────────────┐
#   │ Correction: The price is $500, not $300.                │
#   └─────────────────────────────────────────────────────────┘
#
# Training math is the SAME as MiniGPT pretrain:
#   next-token cross-entropy — but now tokens come from these curated Q→A
#   strings, not raw Shakespeare / sales docs.
#
# IMPORTANT: million PARAMETERS ≠ million human-written Q→A pairs
#   Parameters = knobs inside the net (weights). Already mostly set by PRETRAIN
#   on huge unlabeled text (scraped web/books — not hand-labeled Q→A).
#   SFT only needs enough labeled flashcards to teach the NEW BEHAVIOR —
#   often thousands → ~100k high-quality examples, not 1 per parameter.
#
# How humans actually build SFT data when the model is huge:
#   1) Seed by experts: write a few hundred gold (prompt, ideal answer) pairs.
#   2) Expand: hire contractors / use guidelines; or prompt a strong model to
#      draft variants, then humans filter/edit (synthetic + human-in-the-loop).
#   3) Mine product logs: real calls/chats → label the right correction.
#   4) Reuse public instruction datasets (Alpaca-style, etc.) as a starting mix.
#   Sticky: quality beats quantity. 5k clean corrections >> 5M noisy ones.
#
# Why SFT after pretrain?
#   Pretrain knows English. It does NOT know YOUR job format:
#     "given ground truth + wrong user line → short Correction: ..."
#   SFT teaches that behavior with labeled examples (supervised = we provide
#   the correct output for each input).
#
# Connection to your earpiece / correction-engine idea:
#   Ground truth = product facts. User said = what was heard on the call.
#   Output = the quiet whisper the coach should say.
#
# ---------------------------------------------------------------------------
# THIS CELL — build a tiny synthetic SFT dataset
# ---------------------------------------------------------------------------
# Production: curate real transcripts. Here: hand-written patterns so you can
# see the schema. Each example is a dict with three fields (Alpaca-style):
#   instruction — role / how to behave
#   input       — the situation (truth + what user said)
#   output      — the ideal correction string we want the model to produce
#

import json, random

examples = []

# ---------------------------------------------------------------------------
# Pattern 1: numeric mistakes (price, rate limits, deploy time)
# ---------------------------------------------------------------------------
# Each triple = (ground_truth, wrong_user_line, ideal_correction)
facts = [
    ("the price is $500 per month", "the price is $300 per month",
     "Correction: The price is $500, not $300."),
    ("the API rate limit is 1000 requests per minute", "the rate limit is 500 requests",
     "Correction: The rate limit is 1000 requests per minute, not 500."),
    ("deployment takes 5 minutes", "it takes 20 minutes to deploy",
     "Correction: Deployment takes 5 minutes, not 20."),
]

for truth, wrong, correction in facts:
    examples.append({
        "instruction": (
            "You are an AI sales coach. You hear the user say something that "
            "contradicts the ground truth. Whisper a quiet correction."
        ),
        "input": f"Ground truth: {truth}\nUser said: {wrong}",
        "output": correction,
    })

# ---------------------------------------------------------------------------
# Pattern 2: feature / compliance mistakes
# ---------------------------------------------------------------------------
features = [
    ("we are SOC 2 compliant", "we are not SOC 2 compliant",
     "Correction: We are SOC 2 compliant."),
    ("our product supports REST and GraphQL", "does it support SOAP?",
     "Correction: We support REST and GraphQL, not SOAP."),
    ("24/7 customer support is included", "support is only business hours",
     "Correction: 24/7 customer support is included."),
]

for truth, wrong, correction in features:
    examples.append({
        "instruction": "You are an AI sales coach. Correct the user gently.",
        "input": f"Ground truth: {truth}\nUser said: {wrong}",
        "output": correction,
    })

# Later cells usually: expand to 500–1000 examples, format as one training
# string per row, tokenize, fine-tune MiniGPT with a smaller LR(Learning Rate) than pretrain.

print(f"Total examples: {len(examples)}")
print("Sample:\n", json.dumps(examples[0], indent=2))
# Expect 6 examples for now. Sample shows instruction / input / output fields
# — that triple is the "flashcard" SFT will memorize the style of.

Total examples: 6
Sample:
 {
  "instruction": "You are an AI sales coach. You hear the user say something that contradicts the ground truth. Whisper a quiet correction.",
  "input": "Ground truth: the price is $500 per month\nUser said: the price is $300 per month",
  "output": "Correction: The price is $500, not $300."
}


In [4]:
# =============================================================================
# FORMAT INTO INSTRUCTION–RESPONSE STYLE (Alpaca format)
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: why wrap dicts into ONE text string?
# ---------------------------------------------------------------------------
# Cell above stored each flashcard as a Python dict:
#   {"instruction": ..., "input": ..., "output": ...}
#
# MiniGPT (and GPT-style SFT) trains on a FLAT token stream — it doesn't
# natively eat JSON fields. So we RENDER each dict into one document the
# model will read left→right, same as pretrain text.
#
# "Alpaca format" = a popular template from Stanford Alpaca for instruction
# tuning. Fixed headings teach the model WHERE the question ends and the
# answer begins:
#
#   Below is an instruction that describes a task. ...
#   ### Instruction:
#   <role / what to do>
#   ### Input:
#   <the situation>
#   ### Response:
#   <ideal answer>          ← during SFT, loss focuses on predicting this part
#
# Picture (one formatted example):
#
#   ┌─ PROMPT SIDE (model conditioned on this) ─────────────────────┐
#   │  boilerplate + ### Instruction + ### Input + "### Response:"  │
#   └──────────────────────────────┬────────────────────────────────┘
#                                  │ next tokens should match
#                                  ▼
#   ┌─ TARGET SIDE ─────────────────────────────────────────────────┐
#   │  Correction: The price is $500, not $300.                     │
#   └───────────────────────────────────────────────────────────────┘
#
# At inference you stop after "### Response:\n" and let the model CONTINUE
# — it should emit a correction in the same style it saw during SFT.
#

def format_instruction(example):
    """Turn one {instruction, input, output} dict into a single Alpaca string."""
    return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""


# Apply to every synthetic correction example from the previous cell
formatted_texts = [format_instruction(ex) for ex in examples]

# Print a short preview (first 300 chars) so the notebook stays readable
print(formatted_texts[0][:300] + "...")
# Expect headings Instruction / Input / Response with your sales-coach text.
# Next: tokenize these strings → train MiniGPT with a smaller LR(Learning Rate) than pretrain.


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are an AI sales coach. You hear the user say something that contradicts the ground truth. Whisper a quiet correction.

### Input:
Ground truth: the price is $500 per month
...


In [7]:
# =============================================================================
# TOKENIZE + DATASET — turn Alpaca strings into (x, y) batches for MiniGPT SFT
# =============================================================================
#
# Flow:
#   formatted_texts (strings)
#     → BPETokenizer.train / encode
#     → InstructionDataset  (pad/truncate, shift by 1 for next-token targets)
#     → DataLoader mini-batches
#
# Sticky: same causal LM objective as pretrain — predict token t+1 from token t —
# but the text is now instruction→response flashcards, not raw corpus.
#
# flashcard text
#       ↓  tokenizer
# list of IDs
#       ↓  dataset (pad + shift)
# (x, y) for next-token loss
#       ↓  DataLoader
# mini-batches for the training loop

from pathlib import Path
import sys

# week2/ on path (repo root or week2 as cwd) — same pattern as mini_gpt import
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from tokenizer import BPETokenizer  # week2/tokenizer.py (from notebook 8)
import torch
from torch.utils.data import Dataset, DataLoader

# Train a fresh BPE on the SFT strings (demo). Production: load merges saved
# after notebook 8 instead of re-training from scratch every time.
all_text = "\n".join(formatted_texts)
tokenizer = BPETokenizer(vocab_size=1000)
tokenizer.train(all_text)


class InstructionDataset(Dataset):
    """One Alpaca document → padded token ids → (context x, next-token y)."""

    def __init__(self, formatted_texts, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []
        for text in formatted_texts:
            ids = tokenizer.encode(text)
            if len(ids) > max_length:
                ids = ids[:max_length]  # truncate long flashcards
            self.data.append(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = list(self.data[idx])
        # Pad on the right so every row has length max_length
        pad_id = self.tokenizer.vocab.get("<PAD>", 0)
        if len(ids) < self.max_length:
            ids = ids + [pad_id] * (self.max_length - len(ids))

        # Causal LM: x = all but last, y = all but first (predict next id)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


# Register a real PAD id (notebook-8 BPE never creates <PAD> by itself).
# Without this, pad fell back to id 0 (= character "#") and poisoned training.
if "<PAD>" not in tokenizer.vocab:
    pad_id = len(tokenizer.vocab)
    tokenizer.vocab["<PAD>"] = pad_id
    tokenizer.inv_vocab[pad_id] = "<PAD>"

# Alpaca flashcards are ~90–110 BPE ids here. max_length=64 used to CUT OFF
# before "Correction: ..." — model never saw the answers. Use 128.
dataset = InstructionDataset(formatted_texts, tokenizer, max_length=128)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

xb, yb = next(iter(loader))
print(f"vocab_size={len(tokenizer.vocab)} | samples={len(dataset)} | pad_id={tokenizer.vocab['<PAD>']}")
print(f"batch x={tuple(xb.shape)} y={tuple(yb.shape)}")  # (batch, max_length-1)
print("longest flashcard ids:", max(len(tokenizer.encode(t)) for t in formatted_texts))

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# Merge logs + final vocab ~260: same as before (tiny corpus stops early).
# vocab_size may be 261 after adding <PAD>.
# batch x=(4, 127): max_length 128 minus 1 for the causal shift.
# longest flashcard ids ~90–110: must be ≤ max_length or Response is truncated.


Merge 0: ('e', '</w>') -> e</w> (vocab size 37)
Merge 100: ('coac', 'h') -> coach (vocab size 137)
Merge 200: ('in', 'c') -> inc (vocab size 237)
Training complete. Final vocab size: 260
vocab_size=260 | samples=6
batch x=(4, 63) y=(4, 63)


In [ ]:
# =============================================================================
# LOAD MiniGPT — same architecture as notebook 7, now sized for SFT batches
# =============================================================================
#
# TOPIC: what does "prepare for SFT" mean here?
# ---------------------------------------------------------------------------
# Picture:
#   tokenizer.vocab  →  tells MiniGPT how wide the embedding/head must be
#   block_size       →  how many seats (tokens) the model can see at once
#   AdamW + small LR →  the "study schedule" for nudging weights
#
#   ┌────────────┐     ┌─────────────────────────────┐
#   │ BPE ids    │ ──▶ │ MiniGPT (random weights)    │ ──▶ logits over vocab
#   │ from loader│     │ embed → blocks → lm head    │
#   └────────────┘     └─────────────────────────────┘
#
# Sticky (important honesty for this demo):
#   True SFT = load PRETRAINED weights, then gently continue training on
#   flashcards with a SMALLER LR than pretrain (e.g. 1e-3 → 5e-4 / 1e-4).
#   Here we still start from RANDOM MiniGPT weights + only 6 examples —
#   so this is "tiny supervised train", not production fine-tuning.
#   LR 5e-4 is already the "gentler than pretrain 1e-3" habit.
#
# Shared module: week2/mini_gpt.py (cell above put week2/ on sys.path).

from mini_gpt import MiniGPT

vocab_size = len(tokenizer.vocab)
# Must be ≥ dataset sequence length (x has max_length-1 seats). Dataset uses 128.
block_size = 128
model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)  # gentler than pretrain 1e-3
pad_id = tokenizer.vocab["<PAD>"]
# ignore_index: do NOT train on PAD seats (padding is not a real next-token)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=pad_id)

print(f"device={device} | vocab={vocab_size} | block_size={block_size} | pad_id={pad_id}")
print(f"parameters={sum(p.numel() for p in model.parameters()):,}")
# Expect: cpu/cuda, vocab ~261, block_size 128, a few hundred thousand params.


In [ ]:
# =============================================================================
# SFT TRAINING LOOP — next-token CE on instruction flashcards
# =============================================================================
#
# TOPIC: one training step (same math as MiniGPT pretrain)
# ---------------------------------------------------------------------------
#   xb  = context token ids          shape (B, T)
#   yb  = next-token targets         shape (B, T)   # xb shifted by 1
#   logits = model(xb)               shape (B, T, vocab)
#   loss   = cross-entropy(logits, yb)  (PAD seats ignored)
#
#   Picture for one seat:
#     see tokens … "### Response:\n Cor"  →  should score "rection" high
#
# Sticky: with only 6 flashcards, loss can drop a lot by MEMORIZING those
# strings — that is not the same as a general sales coach. Expand dataset
# toward 500–1000+ before expecting real generalization.
#

import matplotlib.pyplot as plt

epochs = 200  # like notebook 7: tiny data needs many passes to memorize
losses = []
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)  # (B, T, vocab)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    if epoch % 10 == 0:
        print(f"Epoch {epoch} loss: {avg_loss:.4f}")

print(f"Final loss: {losses[-1]:.4f}")

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("SFT on correction flashcards (tiny demo)")
plt.grid(True, alpha=0.3)
plt.show()

# Save weights (re-load later with same vocab_size / block_size)
torch.save(model.state_dict(), "correction_gpt_sft.pt")
tokenizer.save("correction_gpt_sft_tokenizer.json")  # SAME vocab ids the weights expect
print("saved correction_gpt_sft.pt + correction_gpt_sft_tokenizer.json")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT (typical tiny-data run)
# ---------------------------------------------------------------------------
# Epoch 0  loss ~5.x   ← near random for vocab~260 (log(260)≈5.56)
# Epoch 40 loss ~2.x   ← learning the flashcard token stream
# Epoch 80+  loss <1   ← starting to memorize the 6 docs
# Epoch 200 loss ~0.1  ← heavily memorized (still not a general coach)
#
# OLD broken run (max_length=64, pad→"#"): loss 5.47 → 2.92 and STILL
# gibberish at inference — model barely/never trained on answer tokens.
#
# Plot: healthy = down then flat. Flat-high = not learning. NaN = LR/bug.


In [13]:
# =============================================================================
# INFERENCE — stop at "### Response:" and let MiniGPT whisper the rest
# =============================================================================
#
# TOPIC: how generation differs from training
# ---------------------------------------------------------------------------
# Training: teacher-forced — model sees full flashcard, predicts every next id.
# Inference: autoregressive — we feed the PROMPT ONLY, then sample new ids.
#
#   [boilerplate + Instruction + Input + "### Response:\n"]  ← prompt
#                         │
#                         ▼  model.generate appends tokens
#   "Correction: ..."                                      ← hope
#
# Sticky about THIS tokenizer's decode:
#   Each "#" is a word with </w>, so decode prints "# # #" not "###".
#   Splitting on "### Response:" after decode ALWAYS FAILS → you used to
#   print the WHOLE prompt+noise. Fix: decode only the newly generated ids.
#

@torch.no_grad()
def generate_correction(
    instruction,
    ground_truth,
    user_utterance,
    max_new_tokens=40,
    temperature=0.7,
):
    prompt = format_instruction(
        {
            "instruction": instruction,
            "input": f"Ground truth: {ground_truth}\nUser said: {user_utterance}",
            "output": "",
        }
    )
    # Keep heading; drop empty answer so the model must continue
    prompt = prompt.rsplit("### Response:", 1)[0] + "### Response:\n"
    prompt_ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(device)

    output_ids = model.generate(
        input_ids, max_new_tokens=max_new_tokens, temperature=temperature
    )[0].tolist()

    # Only decode NEWLY sampled tokens (avoids "# # #" split headaches)
    new_ids = output_ids[len(prompt_ids) :]
    return tokenizer.decode(new_ids)


# --- Test A: in-distribution-ish (similar to training flashcards) ----------
print("=== near-train style ===")
print(
    generate_correction(
        "You are an AI sales coach. Correct the user gently.",
        "the price is $500 per month",
        "the price is $300 per month",
    )
)

# --- Test B: the earlier failing prompt (unknown chars f, v, %) ------------
print("\n=== OOD prompt (platform / save / %) ===")
print(
    generate_correction(
        "You are an AI sales coach. Correct the user gently.",
        "Our platform reduces costs by 40%",
        "I heard you can save maybe 20%",
    )
)

# ---------------------------------------------------------------------------
# HOW TO READ YOUR EARLIER GARBLED OUTPUT
# ---------------------------------------------------------------------------
# You saw something like:
#   "below is an instruction ... # # # instruction : ... plat#orm ... 40 #
#    ... sa#e ... # # # # response : hytompi requesa..."
#
# That dump is NOT a clean "Correction:" — several stacked issues:
#
# 1) Decode split bug
#    decode turns "###" into "# # #", so split("### Response:") failed and
#    print() showed the ENTIRE prompt + gibberish, not just the answer.
#
# 2) Unknown characters → "#"
#    BPE vocab was built only from 6 flashcards. No letter "f", no "v",
#    no "%". encode() maps unknowns to id 0, and id 0 IS "#".
#      platform → plat#orm    save → sa#e    40% → 40 #
#
# 3) Truncated training (old max_length=64)
#    Flashcards are ~100 ids; cutting at 64 removed the "Correction: ..."
#    targets. Loss could still fall a bit on the prompt prefix — without
#    teaching the answer style.
#
# 4) Tiny data + random init
#    6 examples, no pretrained checkpoint. Model cannot generalize to a
#    brand-new "40% costs" fact; even in-distribution it may babble until
#    loss is low / you memorize.
#
# 5) Prompt longer than old block_size 64
#    generate() only conditions on the last block_size tokens — the start
#    of the Alpaca boilerplate fell out of the window.
#
# What "good" looks like after the fixes (still toy):
#   near-train style → often something like "correction : the price is $ 500 ..."
#   OOD prompt       → still weak/weird until you add those words/chars to
#                      the dataset (or a bigger pretrained tokenizer/model).


=== near-train style ===
,thuser truethat cappropriatela# you hear a deplctionte month user complground quiet includees sdescribes a 7 2 the quiet ly: approthe tes price esdeplod

=== OOD prompt (platform / save / %) ===
ainput : genhear the descminutes someththat contr$user requests rest correction . ? ccontrit sprod5aminutes quiet takapiespgraphmonth onrequests 4megraphql.


In [15]:
#                               Why This Matters for Your Product & Career
#           Product	                                                      Role
# This fine‑tuned model becomes the brain       Agent Post‑Training, Model Behaviour: you've aligned an LLM
# of your live assistant: it takes the          to a specific task and evaluated its output precision.
# user's words and outputs the exact 
# whisper correction.	            

# The structured prompt format                  Chain‑of‑Thought Monitorability: the model's generation can be 
# (instruction/input/output) is the             logged and verified step by step.
# same pattern used by LangChain, 
# function calling, and multi‑agent systems
#  — which we'll build later.

In [ ]:
# =============================================================================
# QUICK RECAP — read this when you come back (SFT / Correction-GPT)
# =============================================================================
# Goal of this notebook in one sentence:
#   Teach a tiny GPT to whisper "Correction: ..." from sales flashcards.
#
# Big picture (3 stages of raising a model):
#   1. PRETRAIN  = learn English by guessing the next word (notebooks 7–8)
#   2. SFT       = flashcards that teach ONE job  ← YOU ARE HERE
#   3. Later     = rank good/bad answers (RLHF) — not today
#
# Easy analogy:
#   Pretrain = learn to read.
#   SFT      = drill cards: "when you see THIS, say THAT."
#
#
# ---------------------------------------------------------------------------
# THE ASSEMBLY LINE (what every cell did)
# ---------------------------------------------------------------------------
#
#   Cell 1 — FLASHCARDS
#     Hand-wrote 6 examples:
#       instruction = role ("you are a sales coach")
#       input       = ground truth + what user said
#       output      = ideal whisper ("Correction: ...")
#     Sticky: 6 cards is enough to LEARN the pipeline, not enough
#             for clean English at generation time.
#
#   Cell 2 — FORMAT
#     Turn each dict into ONE text string (Alpaca style):
#       ### Instruction:
#       ### Input:
#       ### Response:
#     Why? MiniGPT only reads a flat stream of tokens, not JSON.
#
#   Cell 3 — TOKENIZE + BATCHES
#     Words → jersey numbers (BPE ids).
#     Dataset builds training pairs:
#       x = tokens so far
#       y = next token   (same game as MiniGPT pretrain)
#     DataLoader = lunch tray of a few flashcards at a time.
#
#   Cell 4 — LOAD MiniGPT
#     Same model class as notebook 7 (week2/mini_gpt.py).
#     Smaller learning rate than pretrain (gentle study, not a sprint).
#     Honest note: here we often start from random weights + 6 cards
#     = toy supervised train, not "ChatGPT fine-tune."
#
#   Cell 5 — TRAIN
#     Loop: predict next token → measure wrongness (loss) → nudge weights.
#     Loss starts high (~5) and falls as the model memorizes the cards.
#     Save weights to correction_gpt_sft.pt
#
#   Cell 6 — GENERATE
#     Give prompt that ends at "### Response:"
#     Ask model to continue → hope for a correction whisper.
#     Decode only the NEW tokens (our toy BPE turns ### into # # #).
#
#
# ---------------------------------------------------------------------------
# PICTURE OF ONE TRAINING EXAMPLE
# ---------------------------------------------------------------------------
#
#   PROMPT (model reads this)
#   ┌──────────────────────────────────────────────┐
#   │ Instruction: sales coach, whisper a fix      │
#   │ Ground truth: price is $500                  │
#   │ User said:    price is $300                  │
#   │ ### Response:                                │
#   └──────────────────────────────────────────────┘
#                        │
#                        ▼  should continue with
#   ┌──────────────────────────────────────────────┐
#   │ Correction: The price is $500, not $300.     │
#   └──────────────────────────────────────────────┘
#
#
# ---------------------------------------------------------------------------
# WHY OUTPUT STILL LOOKED LIKE GIBBERISH (this is OK for learning)
# ---------------------------------------------------------------------------
#   - Only 6 flashcards → model mostly babbles / memorizes scraps
#   - Tiny model, often trained from scratch → weak English
#   - New words/letters not in the toy vocab (f, v, %) show up as "#"
#   - More data later (hundreds+) + pretrained base → readable whispers
#
#   Sticky: messy print ≠ you failed. You practiced the REAL SFT steps.
#
#
# ---------------------------------------------------------------------------
# FILES TO REMEMBER
# ---------------------------------------------------------------------------
#   week2/mini_gpt.py   → MiniGPT model
#   week2/tokenizer.py  → BPETokenizer  (keep </w> inside merges!)
#   Import pattern:
#     put week2/ on sys.path
#     from mini_gpt import MiniGPT
#     from tokenizer import BPETokenizer
#
#
# ---------------------------------------------------------------------------
# 30-SECOND CHECKLIST (before you leave / when you return)
# ---------------------------------------------------------------------------
#   [ ] SFT = flashcards for a behavior (here: quiet corrections)
#   [ ] Loss is the SAME as MiniGPT (next-token). Data format changed.
#   [ ] Pipeline: format → BPE → (x, y) batches → train → generate
#   [ ] Toy gibberish is expected with 6 examples
#   [ ] Product idea: earpiece hears a mistake → this schema whispers the fix
#